# E07：数组筛选、聚合、缺失值与广播

**课程**：Python 程序设计  
**适用对象**：应用统计学专业一年级  
**实验类型**：验证性实验  
**建议学时**：2 学时

## 实验目标

完成本实验后，你应当能够：

1. 使用 `np.nan`、`np.isnan()` 和布尔掩码处理数值缺失。
2. 组合多个布尔条件完成筛选和分组汇总。
3. 使用二维数组和 `axis` 明确按行与按列的处理方向。
4. 直观使用广播完成按列阈值判断，并核验筛选前后的统计口径。

## 实验信息

- 姓名：  
- 学号：  
- 班级：  
- 日期：

## 提交要求

1. 在本 Notebook 中完成全部任务，不删除题目、提示和课程提供的数据单元。
2. 保留最终可运行的代码、输出结果和必要的 Markdown 说明。
3. 不修改或覆盖 `lab/shared` 中的原始数据和官方检查点。
4. 提交前重启内核并按顺序运行全部单元，检查数组形状、样本量和统计口径。
5. 如使用帮助文档、代码补全或 AI 工具，应在实验小结中说明用途和核验方式。

## Jupyter 使用提示

- 先运行输入准备单元，再完成后续任务。
- 修改代码后必须重新运行当前单元，输出才会更新。
- 数组计算结果异常时，先检查 `shape`、缺失值和布尔掩码。
- 如果变量状态混乱，重启内核并从上到下重新运行。

## 建议节奏

- 保留缺失值并创建对齐数组：15 分钟
- 缺失值识别与有效值聚合：15 分钟
- 布尔筛选与组合条件：15 分钟
- 二维数组和 `axis`：15 分钟
- 广播与综合汇总：15 分钟
- 调试、实验小结与提交：15 分钟

## 主线延续：保持 160 条记录对齐

本实验继续使用与 E06 相同的输入选择和核验规则。与 E06 删除缺失记录不同，E07 使用 `np.nan` 保留原有 160 个位置，使数值数组与记录列表保持对齐。字符串字段不组成混合数组，而是直接转换为同长度的布尔掩码。

输入按以下顺序查找：

1. E05 标准输出目录中的个人成果。
2. 当前工作目录中的 `e05_cleaned_records.json`，按学生自行提供的输入处理，来源需自行确认。
3. 仓库中的官方 E05 检查点。
4. 当前工作目录中的 `e05_cleaned_records_official.json` 官方副本。
5. 联网获取官方检查点，核验通过后按上述官方副本名称缓存到当前工作目录。

将 `USE_STUDENT_E05_OUTPUT` 改为 `False` 时，跳过前两项。只有文件不存在才会继续向后查找；已有文件解析或核验失败会明确报错，不会静默改用官方数据。以准备单元打印的来源、实际路径和当前工作目录为准；若从其他目录启动内核，当前工作目录不一定是 Notebook 所在目录。

准备代码直接运行即可，不要求掌握联网下载实现。单独下载 Notebook 时，可自行提供同名输入或联网取得官方副本；官方副本另取文件名，避免覆盖或误认个人成果。

In [ ]:
import json
from pathlib import Path
from urllib.request import urlopen

import numpy as np

# True：优先使用个人成果；False：跳过个人文件，直接使用官方检查点。
USE_STUDENT_E05_OUTPUT = True

course_lab_dir = Path("..")
if not (course_lab_dir / "chapter2").exists():
    course_lab_dir = Path("lab/applied-stat-64")

shared_dir = Path("../../shared")
if not shared_dir.exists():
    shared_dir = Path("lab/shared")

student_path = course_lab_dir / "chapter2" / "e05_output" / "e05_cleaned_records.json"
current_student_path = Path("e05_cleaned_records.json")
official_path = shared_dir / "checkpoints" / "e05_cleaned_records.json"
# 官方本地副本另取文件名，避免与学生提供的同名输入混淆。
official_cache_path = Path("e05_cleaned_records_official.json")

input_candidates = []
if USE_STUDENT_E05_OUTPUT:
    input_candidates.extend([
        (student_path, "学生 E05 标准输出"),
        (current_student_path, "当前工作目录自行提供的 E05 输入（来源需自行确认）"),
    ])
input_candidates.extend([
    (official_path, "仓库官方 E05 检查点"),
    (official_cache_path, "当前工作目录官方检查点副本"),
])

input_path = None
downloaded_text = None
for candidate_path, candidate_source in input_candidates:
    if candidate_path.exists():
        input_path = candidate_path
        input_source = candidate_source
        break

if input_path is None:
    input_path = official_cache_path
    input_source = "联网获取的官方 E05 检查点"
    official_url = (
        "https://rwang0040.github.io/python-for-statistics-public/"
        "lab/shared/checkpoints/e05_cleaned_records.json"
    )
    print("本地未找到可用路径，尝试联网获取官方检查点：", official_url)
    try:
        with urlopen(official_url, timeout=10) as response:
            downloaded_text = response.read().decode("utf-8")
    except OSError as error:
        raise RuntimeError(
            "无法联网获取官方检查点。请检查网络，或手动下载官方文件，"
            "重命名为 e05_cleaned_records_official.json 后放到当前工作目录。"
        ) from error

print("输入来源：", input_source)
print("输入文件：", input_path.resolve())
print("当前工作目录：", Path.cwd())

# 只有文件不存在才向后查找；已有文件损坏或核验失败时不静默切换来源。
if downloaded_text is None:
    with open(input_path, "r", encoding="utf-8") as file:
        records = json.load(file)
else:
    records = json.loads(downloaded_text)

if not isinstance(records, list) or not all(isinstance(record, dict) for record in records):
    raise ValueError("输入必须是由字典记录组成的列表，请检查以上输入来源和路径。")

expected_fields = [
    "student_id",
    "alias",
    "gender",
    "major",
    "prior_programming",
    "support_plan",
    "reported_study_hours",
    "reported_attendance_rate",
]
student_ids = [record.get("student_id") for record in records]
missing_hours = sum(
    record.get("reported_study_hours") is None
    for record in records
)
missing_attendance = sum(
    record.get("reported_attendance_rate") is None
    for record in records
)

input_checks = {
    "记录数为 160": len(records) == 160,
    "student_id 唯一且非空": (
        all(isinstance(value, str) and bool(value) for value in student_ids)
        and len(set(student_ids)) == len(student_ids)
    ),
    "字段及顺序一致": all(
        list(record.keys()) == expected_fields
        for record in records
    ),
    "学习时长缺失数为 8": missing_hours == 8,
    "出勤率缺失数为 4": missing_attendance == 4,
}

for check_name, passed in input_checks.items():
    print(check_name, "：", passed)

if not all(input_checks.values()):
    raise ValueError(
        "输入核验未通过：请检查以上来源和文件。个人成果有误时请先修正；"
        "如需主动改用官方数据，可把 USE_STUDENT_E05_OUTPUT 改为 False。"
    )

# 下载内容通过同一套核验后才保存；网络错误与本地写入错误分别提示。
if downloaded_text is not None:
    try:
        official_cache_path.write_text(downloaded_text, encoding="utf-8")
    except OSError as error:
        raise RuntimeError(
            "官方数据已获取并通过核验，但无法保存本地副本。"
            "请检查当前工作目录的写入权限或磁盘空间。"
        ) from error
    print("官方检查点已缓存：", official_cache_path.resolve())

## 任务 1：使用 `np.nan` 保留缺失位置

按 `records` 的原顺序，分别从 `reported_study_hours`、`reported_attendance_rate` 创建浮点数组 `study_hours`、`attendance_rates`。字段为 `None` 时，在同一位置写入 `np.nan`，保留全部 160 个位置。

建立同长度的布尔数组 `feedback_mask`，`support_plan == "feedback"` 的位置为 `True`。输出三个数组的形状和前 10 个学习时长。

In [ ]:
# TODO：把 None 转换为 np.nan，建立两个等长浮点数组
# study_hours =
# attendance_rates =

# TODO：建立 support_plan 为 feedback 的布尔掩码
# feedback_mask =

# 课程提供：完成三个数组后，取消注释检查 shape
# print("三个数组 shape：", study_hours.shape, attendance_rates.shape, feedback_mask.shape)

## 任务 2：识别缺失值并计算有效统计量

用 `np.isnan()` 分别建立 `missing_hours_mask` 和 `missing_attendance_mask`，输出两列缺失数。

建立非缺失学习时长掩码 `valid_hours_mask`，筛选为 `valid_hours`。输出有效样本量、均值、最小值、最大值和标准差。标准差采用本讲 NumPy 的默认口径 `std()`（`ddof=0`）。

In [ ]:
# TODO：建立两个缺失掩码
# missing_hours_mask =
# missing_attendance_mask =

# TODO：建立非缺失掩码并筛选有效学习时长
# valid_hours_mask =
# valid_hours =

# 课程提供：完成缺失掩码后，取消注释检查数量
# print("学习时长缺失数：", missing_hours_mask.sum())
# print("出勤率缺失数：", missing_attendance_mask.sum())
# TODO：输出有效样本量、均值、最小值、最大值和标准差

## 任务 3：组合布尔掩码完成分组统计

组合“学习时长非缺失”和支持方案掩码，分别计算 `feedback` 与 `regular` 方案的有效样本量和平均学习时长。

In [ ]:
# TODO：组合非缺失掩码与支持方案掩码
# feedback_valid_mask =
# regular_valid_mask =

# TODO：筛选两组学习时长并输出样本量和均值
# feedback_hours =
# regular_hours =
# print(...)

## 任务 4：用二维数组检查完整记录

按“学习时长、出勤率”两列建立 `numeric_matrix`。使用 `axis=1` 找出任一字段缺失的行，筛选得到 `complete_matrix`；再使用 `axis=0` 计算两列均值。

In [ ]:
# TODO：按“学习时长、出勤率”两列建立二维数组
# numeric_matrix =

# TODO：使用 np.isnan(...).any(axis=1) 找出不完整行，再筛选完整记录
# incomplete_row_mask =
# complete_matrix =

# TODO：使用 axis=0 计算完整记录的两列均值
# print(...)

## 任务 5：直观使用广播完成按列判断

课程提供的阈值数组 `thresholds` 按列对应每周学习 8 小时和出勤率 80%。将 `complete_matrix` 与阈值数组逐项作大于等于比较，保存为 `meets_thresholds`。

使用 `all(axis=1)` 判断每行是否同时达到两个阈值，保存为 `meets_both`。输出完整记录、阈值和比较结果的形状，并统计 `meets_both` 中为 `True` 的记录数。

In [ ]:
# 课程提供：两列的参考阈值，顺序为学习时长、出勤率
thresholds = np.array([8.0, 80.0])

# TODO：利用广播逐列比较完整记录和阈值
# meets_thresholds =

# TODO：使用 all(axis=1) 判断每行是否同时达到两个阈值
# meets_both =

# 课程提供：完成比较后，取消注释检查形状
# print("完整记录 shape：", complete_matrix.shape)
# print("阈值 shape：", thresholds.shape)
# print("比较结果 shape：", meets_thresholds.shape)

## 任务 6：完成综合口径核验

在课程提供的 `summary` 中补全以下键，并输出摘要：

| 键名 | 内容 |
| --- | --- |
| `record_count` | 原始记录数 |
| `missing_study_hours` | 学习时长缺失数 |
| `missing_attendance` | 出勤率缺失数 |
| `complete_record_count` | 两个数值字段都完整的记录数 |
| `feedback_valid_count` | `feedback` 组有效学习时长数 |
| `regular_valid_count` | `regular` 组有效学习时长数 |
| `both_threshold_count` | 同时达到任务 5 两个阈值的记录数 |

核对两组有效样本量之和是否等于 `valid_hours.size`。在实验小结中说明学习时长和出勤率的单位差异，以及直接把它们按行求平均的问题。

In [ ]:
# 课程提供：摘要字典骨架
summary = {
    "record_count": len(records),
    # TODO：补全缺失数、完整记录数、两组有效样本量和双阈值人数
}

# TODO：补全字典后输出摘要，并核验两组有效样本量之和
# print(summary)
# 两组有效样本量之和应等于全部有效学习时长数

## 实验小结

请用 Markdown 回答：

1. 为什么使用 `np.nan` 后仍要同时报告有效样本量？
2. `axis=0` 和 `axis=1` 在本实验中分别沿什么方向处理？
3. 为什么不能把学习时长和出勤率直接按行求平均？

### 实验小结

- 